#Set-Up

In [ ]:
import os, re
import torch, matplotlib
import math
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
import nimblephysics
from types import SimpleNamespace
import torch.nn.functional as Fnn

In [ ]:
!git clone https://github.com/stanfordnmbl/GaitDynamics.git

print("Setup complete")

In [ ]:
FEATURE_COLS = ["left-hip-x","right-hip-x","left-knee-x","right-knee-x","left-ankle-x","right-ankle-x"]

CFG = {
    "window": 600,
    "d_model": 128,
    "n_layers": 4,
    "cond_dim": 64,
    "t_steps": 1000,
    "t_embed_dim": 128
}

# Encoders

In [ ]:
class SubjectEncoder(nn.Module):  # Encode the Subject Data
  def __init__(self, input_dim, emb_dim=64, hidden_dim=128):
    super().__init__()
    self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True, bidirectional=True)
    self.proj = nn.Linear(hidden_dim * 2, emb_dim)

  def forward(self, x):
    _, (h, _) = self.lstm(x)
    h = torch.cat([h[0], h[1]], dim=-1)
    z = self.proj(h)
    z = Fnn.normalize(z, dim=-1)
    return z


class CondEncoder(nn.Module): # Encode Speed Data and Combine with Subject Encoder
  def __init__(self, input_dim, emb_dim=64):
    super().__init__()
    self.subj_encoder = SubjectEncoder(input_dim, emb_dim)

    self.speed_mlp = nn.Sequential(
      nn.Linear(4, emb_dim),
      nn.SiLU(),
      nn.Linear(emb_dim, emb_dim)
    )

    self.proj = nn.Linear(emb_dim * 2, emb_dim)

  def forward(self, x_seq, speed_src, speed_tgt, return_parts=False):
    s_emb = self.subj_encoder(x_seq)

    v_src = speed_src.view(-1, 1)
    v_tgt = speed_tgt.view(-1, 1)
    v_delta = v_tgt - v_src
    v_ratio = v_tgt / (v_src + 1e-6)

    speed_feats = torch.cat([v_src, v_tgt, v_delta, v_ratio], dim=-1)
    v_emb = self.speed_mlp(speed_feats)

    cond = torch.cat([s_emb, v_emb], dim=-1)
    cond = self.proj(cond)
    if return_parts:
      return cond, {"s_emb": s_emb, "v_emb": v_emb}
    return cond

# FiLM and Predictor

In [ ]:
class FiLM(nn.Module): # Feature-wise Linear Modulation
  def __init__(self, d_model, cond_dim):
    super().__init__()
    self.mlp = nn.Sequential(nn.SiLU(), nn.Linear(cond_dim, 2*d_model))

  def forward(self, h, cond):
    gamma, beta = self.mlp(cond).chunk(2, dim=-1)
    return (1 + gamma)[:, None, :] * h + beta[:, None, :]

class GDEncoderWrapper(nn.Module): # Wraps the gait dynamics Transformer encoder architecture to produce kinematics
  def __init__(self, repr_dim, gd_opt, nlayers=4):
    super().__init__()
    self.gd_enc = TransformerEncoderArchitecture(
      repr_dim=repr_dim,
      opt=gd_opt,
      nlayers=nlayers
    )
    self.embedding_dim = 192

  def forward(self, x):
    sequence = self.gd_enc.input_to_embedding(x)
    sequence = self.gd_enc.encoder_layers(sequence)
    return sequence

def sinusoidal_timestep_embedding(t, dim, max_period=10000): # Creates sinusoidal embeddings to encode the current diffusion timestep
  half = dim // 2
  freqs = torch.exp(
      -math.log(max_period) * torch.arange(0, half, device=t.device, dtype=torch.float32) / half
  )
  args = t[:, None].float() * freqs[None]
  emb = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
  if dim % 2 == 1:
    emb = torch.cat([emb, torch.zeros_like(emb[:, :1])], dim=-1)
  return emb

class EpsPredictor(nn.Module): # Predicts the diffusion noise from residual/source kinematics conditioned on subject, speed, and timestep information.
  def __init__(self, in_dim, out_dim, cond_dim, t_embed_dim, gd_opt, n_layers=4):
    super().__init__()
    self.in_dim = in_dim
    self.out_dim = out_dim

    self.enc = GDEncoderWrapper(repr_dim=in_dim, gd_opt=gd_opt, nlayers=n_layers)
    self._enc_forward = lambda x: self.enc(x)

    self.t_mlp = nn.Sequential(
      nn.Linear(t_embed_dim, cond_dim),
      nn.SiLU(),
      nn.Linear(cond_dim, cond_dim),
    )

    self.film = FiLM(self.enc.embedding_dim, cond_dim)
    self.out = nn.Linear(self.enc.embedding_dim, out_dim)

  def forward(self, x_in, cond_vec, t):
    t_emb = sinusoidal_timestep_embedding(t, dim=CFG["t_embed_dim"])
    t_emb = self.t_mlp(t_emb)
    cond = cond_vec + t_emb

    h = self._enc_forward(x_in)
    h = self.film(h, cond)
    eps_hat = self.out(h)
    return eps_hat

# VP Scheduler

In [ ]:
class VPScheduler: # Defines the variance-preserving diffusion noise schedule used to compute alpha and sigma at each timestep
  def __init__(self, T=1000, beta_start=1e-4, beta_end=0.02):
    self.T = T
    betas = torch.linspace(beta_start, beta_end, T)
    alphas = 1.0 - betas
    self.alphas_cumprod = torch.cumprod(alphas, dim=0)  # [T]
    self.betas = betas
  def alpha(self, t):
    return self.alphas_cumprod.to(t.device)[t]**0.5
  def sigma(self, t):
    return (1 - self.alphas_cumprod.to(t.device)[t])**0.5
scheduler = VPScheduler(T=CFG["t_steps"])

# Build Model

In [ ]:
from model.model import TransformerEncoderArchitecture

# Builds the diffusion noise-prediction model and conditioning encoder

F = len(FEATURE_COLS)

in_feature_cols = [f"res_{c}" for c in FEATURE_COLS] + [f"src_{c}" for c in FEATURE_COLS]
in_dim = len(in_feature_cols)
out_dim = F

kinematic_diffusion_col_loc = list(range(in_dim))

gd_opt = SimpleNamespace(
    window_len=CFG["window"],
    kinematic_diffusion_col_loc=kinematic_diffusion_col_loc,
    model_states_column_names=in_feature_cols,
)

model = EpsPredictor(
    in_dim=in_dim,
    out_dim=out_dim,
    cond_dim=CFG["cond_dim"],
    t_embed_dim=CFG["t_embed_dim"],
    gd_opt=gd_opt,
    n_layers=4
).to(device)

cond_enc = CondEncoder(input_dim=F, emb_dim=CFG["cond_dim"]).to(device)